# Message Passing (Pooling) Experiments - Standalone Environment
Run the cells below sequentially. This notebook is specifically designed to compare different message passing (readout) mechanisms: max, mean, add, and attention-based pooling.

In [ ]:
!mkdir -p src
%%writefile src/utils.py
import yaml
import torch

def load_config(config_path):
    with open(config_path, 'r') as f:
        return yaml.safe_load(f)

def get_device(device_str):
    if device_str == "auto":
        if torch.cuda.is_available():
            return torch.device('cuda')
        elif torch.backends.mps.is_available():
            return torch.device('mps')
        else:
            return torch.device('cpu')
    return torch.device(device_str)


In [ ]:
!mkdir -p src
%%writefile src/data.py
import os
import os.path as osp
from torch_geometric.datasets import UPFD
from torch_geometric.loader import DataLoader
from torch_geometric.transforms import ToUndirected

def get_loaders(config):
    """
    Creates train, val, and test data loaders based on the provided configuration.
    """
    data_cfg = config['data']
    path = osp.join(osp.dirname(osp.realpath(__file__)), '..', data_cfg['data_dir'])
    
    # Common transform to ensure graphs are undirected
    transform = ToUndirected()

    # Use the standard UPFD dataset, which will automatically download if missing
    train_dataset = UPFD(path, data_cfg['dataset'], data_cfg['feature'], 'train', transform)
    val_dataset = UPFD(path, data_cfg['dataset'], data_cfg['feature'], 'val', transform)
    test_dataset = UPFD(path, data_cfg['dataset'], data_cfg['feature'], 'test', transform)

    train_loader = DataLoader(train_dataset, batch_size=data_cfg['batch_size'], shuffle=True)
    val_loader = DataLoader(val_dataset, batch_size=data_cfg['batch_size'], shuffle=False)
    test_loader = DataLoader(test_dataset, batch_size=data_cfg['batch_size'], shuffle=False)

    return train_loader, val_loader, test_loader, train_dataset


In [ ]:
!mkdir -p src
%%writefile src/trainer.py
import torch
import torch.nn.functional as F
from tqdm import tqdm

class Trainer:
    def __init__(self, model, optimizer, device, train_loader, val_loader, test_loader):
        self.model = model.to(device)
        self.optimizer = optimizer
        self.device = device
        self.train_loader = train_loader
        self.val_loader = val_loader
        self.test_loader = test_loader

    def _get_model_kwargs(self, data):
        kwargs = {
            'x': data.x,
            'edge_index': data.edge_index,
            'batch': data.batch
        }
        if hasattr(data, 'sentiment'):
            kwargs['sentiment_features'] = data.sentiment
        return kwargs

    def train_epoch(self):
        self.model.train()
        total_loss = 0
        for data in self.train_loader:
            data = data.to(self.device)
            self.optimizer.zero_grad()
            out = self.model(**self._get_model_kwargs(data))
            loss = F.nll_loss(out, data.y)
            loss.backward()
            self.optimizer.step()
            total_loss += float(loss) * data.num_graphs
        return total_loss / len(self.train_loader.dataset)

    @torch.no_grad()
    def test(self, loader):
        self.model.eval()
        total_correct = total_examples = 0
        for data in loader:
            data = data.to(self.device)
            out = self.model(**self._get_model_kwargs(data))
            pred = out.argmax(dim=-1)
            total_correct += int((pred == data.y).sum())
            total_examples += data.num_graphs
        return total_correct / total_examples

    def fit(self, epochs):
        best_val_acc = 0
        best_test_acc = 0
        for epoch in range(1, epochs + 1):
            loss = self.train_epoch()
            train_acc = self.test(self.train_loader)
            val_acc = self.test(self.val_loader)
            test_acc = self.test(self.test_loader)
            
            if val_acc > best_val_acc:
                best_val_acc = val_acc
                best_test_acc = test_acc
                
            print(f'Epoch: {epoch:02d}, Loss: {loss:.4f}, Train: {train_acc:.4f}, '
                  f'Val: {val_acc:.4f}, Test: {test_acc:.4f}')
        return best_test_acc

class TransductiveTrainer:
    def __init__(self, model, optimizer, device, giant_batch, H, train_idx, val_idx, test_idx, y):
        self.model = model.to(device)
        self.optimizer = optimizer
        self.device = device
        self.giant_batch = giant_batch.to(device)
        self.H = H.to(device)
        self.train_idx = train_idx.to(device)
        self.val_idx = val_idx.to(device)
        self.test_idx = test_idx.to(device)
        self.y = y.to(device)

    def train_epoch(self):
        self.model.train()
        self.optimizer.zero_grad()
        # Forward pass on the entire hypergraph
        out = self.model(
            x=self.giant_batch.x,
            edge_index=self.giant_batch.edge_index,
            batch=self.giant_batch.batch,
            H=self.H
        )
        # Compute loss on training nodes only
        loss = F.nll_loss(out[self.train_idx], self.y[self.train_idx])
        loss.backward()
        self.optimizer.step()
        return float(loss)

    @torch.no_grad()
    def test(self, indices):
        self.model.eval()
        out = self.model(
            x=self.giant_batch.x,
            edge_index=self.giant_batch.edge_index,
            batch=self.giant_batch.batch,
            H=self.H
        )
        pred = out[indices].argmax(dim=-1)
        correct = int((pred == self.y[indices]).sum())
        return correct / len(indices)

    def fit(self, epochs):
        best_val_acc = 0
        best_test_acc = 0
        for epoch in range(1, epochs + 1):
            loss = self.train_epoch()
            train_acc = self.test(self.train_idx)
            val_acc = self.test(self.val_idx)
            test_acc = self.test(self.test_idx)
            
            if val_acc > best_val_acc:
                best_val_acc = val_acc
                best_test_acc = test_acc
                
            print(f'Epoch: {epoch:02d}, Loss: {loss:.4f}, Train: {train_acc:.4f}, '
                  f'Val: {val_acc:.4f}, Test: {test_acc:.4f}')
        return best_test_acc




In [ ]:
!mkdir -p src/models
%%writefile src/models/classifier.py
import torch.nn as nn
from torch.nn import Linear

class MLPClassifier(nn.Module):
    def __init__(self, in_channels, out_channels):
        super().__init__()
        # A simple MLP head. This can be made more complex if needed.
        self.lin = Linear(in_channels, out_channels)

    def forward(self, x):
        h = self.lin(x)
        return h.log_softmax(dim=-1)


In [ ]:
!mkdir -p src/models
%%writefile src/models/text_encoders.py
import torch
from torch.nn import Linear

class TextEncoder(torch.nn.Module):
    def __init__(self, in_channels, hidden_channels):
        super().__init__()
        self.lin = Linear(in_channels, hidden_channels)

    def forward(self, x, batch):
        # Get the root node (news content) features of each graph:
        # In UPFD, the first node of each graph in the batch is the root node.
        # However, to be robust across batches, we find the first occurrence of each batch index.
        
        # This logic identifies the indices of the first node for each graph in the batch
        root_indices = (batch[1:] - batch[:-1]).nonzero(as_tuple=False).view(-1)
        root_indices = torch.cat([root_indices.new_zeros(1), root_indices + 1], dim=0)
        
        news_features = x[root_indices]
        return self.lin(news_features).relu()


In [ ]:
!mkdir -p src/models
%%writefile src/models/gnn_encoders.py
import torch
from torch_geometric.nn import (
    GATConv, GCNConv, SAGEConv,
    global_max_pool, global_mean_pool, global_add_pool, AttentionalAggregation
)

class GNNEncoder(torch.nn.Module):
    def __init__(self, model_type, in_channels, hidden_channels, concat=True, pooling='max'):
        super().__init__()
        self.concat = concat
        self.pooling = pooling
        if model_type == 'GCN':
            self.conv = GCNConv(in_channels, hidden_channels)
        elif model_type == 'SAGE':
            self.conv = SAGEConv(in_channels, hidden_channels)
        elif model_type == 'GAT':
            self.conv = GATConv(in_channels, hidden_channels)
        else:
            raise ValueError(f"Unsupported GNN model type: {model_type}")

        if self.pooling == 'attention':
            gate_nn = torch.nn.Linear(hidden_channels, 1)
            self.pool = AttentionalAggregation(gate_nn)
        elif self.pooling not in ['max', 'mean', 'add']:
            raise ValueError(f"Unsupported pooling type: {self.pooling}")

        if self.concat:
            self.lin0 = torch.nn.Linear(in_channels, hidden_channels)
            self.lin1 = torch.nn.Linear(hidden_channels * 2, hidden_channels)

    def forward(self, x, edge_index, batch):
        h = self.conv(x, edge_index).relu()
        if self.pooling == 'max':
            h = global_max_pool(h, batch)
        elif self.pooling == 'mean':
            h = global_mean_pool(h, batch)
        elif self.pooling == 'add':
            h = global_add_pool(h, batch)
        elif self.pooling == 'attention':
            h = self.pool(h, batch)

        if self.concat:
            # Get the root node (news content) features of each graph
            root_indices = (batch[1:] - batch[:-1]).nonzero(as_tuple=False).view(-1)
            root_indices = torch.cat([root_indices.new_zeros(1), root_indices + 1], dim=0)
            news = x[root_indices]
            news = self.lin0(news).relu()
            h = torch.cat([h, news], dim=1)
            h = self.lin1(h).relu()
            
        return h


In [ ]:
!mkdir -p src/models
%%writefile src/models/hgfnd.py
import math
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.nn.parameter import Parameter
from torch_geometric.nn import (
    SAGEConv, GCNConv, GATConv,
    global_max_pool, global_mean_pool, global_add_pool, AttentionalAggregation
)

class HyperGraphAttentionLayerSparse(nn.Module):
    def __init__(self, in_features, out_features, dropout, alpha, transfer, concat=True, bias=False):
        super(HyperGraphAttentionLayerSparse, self).__init__()
        self.dropout = dropout
        self.in_features = in_features
        self.out_features = out_features
        self.alpha = alpha
        self.concat = concat

        self.transfer = transfer

        if self.transfer:
            self.weight = Parameter(torch.Tensor(self.in_features, self.out_features))
        else:
            self.register_parameter('weight', None)

        self.weight2 = Parameter(torch.Tensor(self.in_features, self.out_features))
        self.weight3 = Parameter(torch.Tensor(self.out_features, self.out_features))

        if bias:
            self.bias = Parameter(torch.Tensor(self.out_features))
        else:
            self.register_parameter('bias', None)

        self.word_context = nn.Embedding(1, self.out_features)

        self.a = nn.Parameter(torch.zeros(size=(2 * out_features, 1)))
        self.a2 = nn.Parameter(torch.zeros(size=(2 * out_features, 1)))
        self.leakyrelu = nn.LeakyReLU(self.alpha)

        self.reset_parameters()

    def reset_parameters(self):
        stdv = 1. / math.sqrt(self.out_features)
        if self.weight is not None:
            self.weight.data.uniform_(-stdv, stdv)
        self.weight2.data.uniform_(-stdv, stdv)
        self.weight3.data.uniform_(-stdv, stdv)
        if self.bias is not None:
            self.bias.data.uniform_(-stdv, stdv)

        nn.init.uniform_(self.a.data, -stdv, stdv)
        nn.init.uniform_(self.a2.data, -stdv, stdv)
        nn.init.uniform_(self.word_context.weight.data, -stdv, stdv)

    def forward(self, x, adj):
        x_4att = x.matmul(self.weight2)

        if self.transfer:
            x = x.matmul(self.weight)
            if self.bias is not None:
                x = x + self.bias

        N1 = adj.shape[1]  # number of edge
        N2 = adj.shape[2]  # number of node

        pair = adj.nonzero().t()

        get = lambda i: x_4att[i][adj[i].nonzero().t()[1]]
        x1 = torch.cat([get(i) for i in torch.arange(x.shape[0]).long()])

        q1 = self.word_context.weight[0:].view(1, -1).repeat(x1.shape[0], 1).view(x1.shape[0], self.out_features)

        pair_h = torch.cat((q1, x1), dim=-1)
        pair_e = self.leakyrelu(torch.matmul(pair_h, self.a).squeeze()).t()
        assert not torch.isnan(pair_e).any()
        pair_e = F.dropout(pair_e, self.dropout, training=self.training)

        e = torch.sparse_coo_tensor(pair, pair_e, torch.Size([x.shape[0], N1, N2])).to_dense()

        zero_vec = -9e15 * torch.ones_like(e)
        attention = torch.where(adj > 0, e, zero_vec)

        attention_edge = F.softmax(attention, dim=2)

        edge = torch.matmul(attention_edge, x)

        edge = F.dropout(edge, self.dropout, training=self.training)

        edge_4att = edge.matmul(self.weight3)

        get = lambda i: edge_4att[i][adj[i].nonzero().t()[0]]
        y1 = torch.cat([get(i) for i in torch.arange(x.shape[0]).long()])

        get = lambda i: x_4att[i][adj[i].nonzero().t()[1]]
        q1 = torch.cat([get(i) for i in torch.arange(x.shape[0]).long()])

        pair_h = torch.cat((q1, y1), dim=-1)
        pair_e = self.leakyrelu(torch.matmul(pair_h, self.a2).squeeze()).t()
        assert not torch.isnan(pair_e).any()
        pair_e = F.dropout(pair_e, self.dropout, training=self.training)

        e = torch.sparse_coo_tensor(pair, pair_e, torch.Size([x.shape[0], N1, N2])).to_dense()

        zero_vec = -9e15 * torch.ones_like(e)
        attention = torch.where(adj > 0, e, zero_vec)

        attention_node = F.softmax(attention.transpose(1, 2), dim=2)

        node = torch.matmul(attention_node, edge)

        if self.concat:
            node = F.elu(node)

        return node, edge  # edge_4att

class HGNN_ATT(nn.Module):
    def __init__(self, input_size, n_hid, output_size, dropout=0.3):
        super(HGNN_ATT, self).__init__()
        self.dropout = dropout
        self.gat1 = HyperGraphAttentionLayerSparse(input_size, n_hid, dropout=self.dropout, alpha=0.2, transfer=False, concat=True)
        self.gat2 = HyperGraphAttentionLayerSparse(n_hid, output_size, dropout=self.dropout, alpha=0.2, transfer=True, concat=False)

    def forward(self, x, H):
        x0 = x
        x, e = self.gat1(x, H)
        x = x + x0
        x = F.dropout(x, self.dropout, training=self.training)
        x1 = x
        x, e = self.gat2(x, H)
        x = x + x1
        return x, e

class NewsHypergraph(nn.Module):
    def __init__(self, hidden_size, n_categories, dropout=0.3):
        super(NewsHypergraph, self).__init__()
        self.hidden_size = hidden_size
        self.n_categories = n_categories
        self.dropout = dropout
        self.hgnn = HGNN_ATT(self.hidden_size, self.hidden_size, self.hidden_size, dropout=self.dropout)

    def forward(self, nodes, HT):
        hypergraph, edge_att = self.hgnn(nodes, HT)
        return hypergraph, edge_att

class HGFND(nn.Module):
    def __init__(self, config: dict, in_channels: int, out_channels: int):
        super().__init__()
        model_cfg = config['model']
        self.gnn_type = model_cfg.get('gnn_type', 'SAGE')
        self.hidden_channels = model_cfg.get('hidden_channels', 128)
        self.dropout = model_cfg.get('dropout', 0.3)
        self.out_channels = out_channels
        self.in_channels = in_channels
        self.pooling = model_cfg.get('pooling', 'max')
        
        if self.gnn_type == 'SAGE':
            self.conv1 = SAGEConv(self.in_channels, self.hidden_channels)
        elif self.gnn_type == 'GCN':
            self.conv1 = GCNConv(self.in_channels, self.hidden_channels)
        elif self.gnn_type == 'GAT':
            self.conv1 = GATConv(self.in_channels, self.hidden_channels)
        else:
            raise ValueError(f"Unsupported GNN type: {self.gnn_type}")
            
        self.lin0 = nn.Linear(self.in_channels, self.hidden_channels)
        self.lin1 = nn.Linear(2 * self.hidden_channels, self.hidden_channels)
        self.cls = nn.Linear(self.hidden_channels, self.out_channels, bias=True)
        
        if self.pooling == 'attention':
            gate_nn = nn.Linear(self.hidden_channels, 1)
            self.pool = AttentionalAggregation(gate_nn)
        elif self.pooling not in ['max', 'mean', 'add']:
            raise ValueError(f"Unsupported pooling type: {self.pooling}")
        
        self.hypergraph_model = NewsHypergraph(self.hidden_channels, self.out_channels, self.dropout)

    def forward(self, x, edge_index, batch, H):
        root = (batch[1:] - batch[:-1]).nonzero(as_tuple=False).view(-1)
        root = torch.cat([root.new_zeros(1), root + 1], dim=0)
        news = x[root]
        news = self.lin0(news).relu()

        p = self.conv1(x, edge_index).relu()
        if self.pooling == 'max':
            p = global_max_pool(p, batch)
        elif self.pooling == 'mean':
            p = global_mean_pool(p, batch)
        elif self.pooling == 'add':
            p = global_add_pool(p, batch)
        elif self.pooling == 'attention':
            p = self.pool(p, batch)
        p = self.lin1(torch.cat([news, p], dim=-1)).relu()

        v = p.unsqueeze(0)
        
        # H is of shape (N, M). 
        # The layer expects adj of shape (batch, M, N), so we do H.t().unsqueeze(0)
        HT = H.t().unsqueeze(0)
        
        v, e = self.hypergraph_model(v, HT)
        result = v.squeeze(0)
        
        return self.cls(result).log_softmax(dim=-1)


In [ ]:
!mkdir -p src/models
%%writefile src/models/upfd_model.py
import torch
import torch.nn as nn
from src.models.gnn_encoders import GNNEncoder
from src.models.text_encoders import TextEncoder
from src.models.classifier import MLPClassifier

class SentimentEncoder(nn.Module):
    def __init__(self, in_channels, hidden_channels):
        super().__init__()
        # assuming sentiment feature is provided separately (e.g., shape [batch_size, in_channels])
        self.lin = nn.Linear(in_channels, hidden_channels)

    def forward(self, sentiment_features):
        return self.lin(sentiment_features).relu()

class UPFDModel(nn.Module):
    def __init__(self, config, in_channels, out_channels):
        super().__init__()
        model_cfg = config['model']
        self.use_hgfnd = model_cfg.get('use_hgfnd', False)
        
        if self.use_hgfnd:
            from src.models.hgfnd import HGFND
            self.hgfnd_model = HGFND(config, in_channels, out_channels)
            return

        self.use_gnn = model_cfg.get('use_gnn', True)
        self.use_text = model_cfg.get('use_text', True)
        self.use_sentiment = model_cfg.get('use_sentiment', False)
        
        hidden_channels = model_cfg['hidden_channels']

        self.gnn_encoder = None
        self.text_encoder = None
        self.sentiment_encoder = None
        combined_channels = 0

        if self.use_gnn:
            self.gnn_encoder = GNNEncoder(
                model_cfg['gnn_type'], in_channels, hidden_channels,
                pooling=model_cfg.get('pooling', 'max')
            )
            combined_channels += hidden_channels

        if self.use_text:
            self.text_encoder = TextEncoder(in_channels, hidden_channels)
            combined_channels += hidden_channels

            # Channels remain the same after co-attention since we still concat them

        if self.use_sentiment:
            sentiment_dim = model_cfg.get('sentiment_dim', 1) # default to 1D sentiment score
            self.sentiment_encoder = SentimentEncoder(sentiment_dim, hidden_channels)
            combined_channels += hidden_channels

        if combined_channels == 0:
            raise ValueError("At least one encoder (GNN, Text, or Sentiment) must be enabled in the config.")

        self.classifier = MLPClassifier(combined_channels, out_channels)

    def forward(self, x, edge_index, batch, sentiment_features=None, H=None):
        if self.use_hgfnd:
            return self.hgfnd_model(x, edge_index, batch, H)

        embeddings = []
        h_gnn = None
        h_text = None

        if self.use_gnn:
            h_gnn = self.gnn_encoder(x, edge_index, batch)
            
        if self.use_text:
            h_text = self.text_encoder(x, batch)

        if h_gnn is not None:
            embeddings.append(h_gnn)
        if h_text is not None:
            embeddings.append(h_text)

        if self.use_sentiment:
            if sentiment_features is None:
                # Fallback to zeros if not provided in the batch
                sentiment_features = torch.zeros(batch.max().item() + 1, self.sentiment_encoder.lin.in_features).to(x.device)
            h_sent = self.sentiment_encoder(sentiment_features)
            embeddings.append(h_sent)

        # Concatenate embeddings from all active encoders
        if len(embeddings) > 1:
            h = torch.cat(embeddings, dim=-1)
        else:
            h = embeddings[0]

        return self.classifier(h)



In [ ]:
%%writefile main.py
import torch
import os.path as osp
from torch_geometric.datasets import UPFD
from torch_geometric.transforms import ToUndirected
from torch_geometric.data import Batch
from src.utils import load_config, get_device
from src.data import get_loaders
from src.models.upfd_model import UPFDModel
from src.trainer import Trainer, TransductiveTrainer

def build_hypergraph(train_dataset, val_dataset, test_dataset, config):
    """
    Builds the global hypergraph incidence matrix H representing news relations.
    1. User Hyperedges: Shared user features across graphs.
    2. Time Hyperedges: Round relative creation time to proximal bins.
    3. Entity Hyperedges: K-Means clustering of news content.
    """
    import numpy as np
    from sklearn.cluster import KMeans
    
    all_graphs = list(train_dataset) + list(val_dataset) + list(test_dataset)
    N = len(all_graphs)
    print(f"Building hypergraph with N={N} nodes (graphs)...")
    
    # 1. User Hyperedges
    user_to_graphs = {}
    for g_idx, data in enumerate(all_graphs):
        # Exclude the root node (news content node) from user matching
        user_features = data.x[1:]
        for i in range(user_features.size(0)):
            feat = tuple(np.round(user_features[i].numpy(), 6))
            if feat not in user_to_graphs:
                user_to_graphs[feat] = set()
            user_to_graphs[feat].add(g_idx)
            
    # Filter users to build hyperedges (keep users who shared at least TWO graphs)
    user_hyperedges = [list(graphs) for graphs in user_to_graphs.values() if len(graphs) >= 2]
    print(f"Constructed {len(user_hyperedges)} user-based hyperedges (size >= 2).")
    
    # 2. Time Hyperedges
    # Load profile features for temporal information to ensure feature-agnostic robust creation
    path = osp.join(osp.dirname(osp.realpath(__file__)), config['data']['data_dir'])
    p_train = UPFD(path, config['data']['dataset'], 'profile', 'train')
    p_val = UPFD(path, config['data']['dataset'], 'profile', 'val')
    p_test = UPFD(path, config['data']['dataset'], 'profile', 'test')
    all_p_graphs = list(p_train) + list(p_val) + list(p_test)
    
    time_decimals = config['model'].get('time_decimals', 2)
    time_to_graphs = {}
    for g_idx, p_data in enumerate(all_p_graphs):
        if p_data.x.size(1) > 9:
            time_vals = p_data.x[1:, 9].numpy()
            for t in time_vals:
                t_rounded = np.round(t, time_decimals)
                if t_rounded not in time_to_graphs:
                    time_to_graphs[t_rounded] = set()
                time_to_graphs[t_rounded].add(g_idx)
                
    time_hyperedges = [list(graphs) for graphs in time_to_graphs.values() if len(graphs) >= 2]
    print(f"Constructed {len(time_hyperedges)} time-based hyperedges (size >= 2).")
    
    # 3. Entity Hyperedges
    root_features = []
    for data in all_graphs:
        root_features.append(data.x[0].numpy())
    root_features = np.vstack(root_features)
    
    num_clusters = config['model'].get('entity_clusters', 50)
    kmeans = KMeans(n_clusters=min(num_clusters, N), random_state=42, n_init='auto')
    cluster_labels = kmeans.fit_predict(root_features)
    
    entity_to_graphs = {}
    for g_idx, label in enumerate(cluster_labels):
        if label not in entity_to_graphs:
            entity_to_graphs[label] = set()
        entity_to_graphs[label].add(g_idx)
        
    entity_hyperedges = [list(graphs) for graphs in entity_to_graphs.values() if len(graphs) >= 2]
    print(f"Constructed {len(entity_hyperedges)} entity-based hyperedges (size >= 2).")
    
    # Combine hyperedges
    all_hyperedges = user_hyperedges + time_hyperedges + entity_hyperedges
    M = len(all_hyperedges)
    
    H = torch.zeros((N, M), dtype=torch.float)
    for h_idx, graphs in enumerate(all_hyperedges):
        for g_idx in graphs:
            H[g_idx, h_idx] = 1.0
            
    print(f"Global incidence matrix shape: {H.shape}")
    return H

def run_experiment(config):
    # Get device
    device = get_device(config['training']['device'])
    print(f"Using device: {device}")

    # Prepare data
    train_loader, val_loader, test_loader, train_dataset = get_loaders(config)
    
    model_cfg = config['model']
    use_hgfnd = model_cfg.get('use_hgfnd', False)
    
    if use_hgfnd:
        print("Initializing HGFND transductive experiment...")
        path = osp.join(osp.dirname(osp.realpath(__file__)), config['data']['data_dir'])
        val_dataset = UPFD(path, config['data']['dataset'], config['data']['feature'], 'val', ToUndirected())
        test_dataset = UPFD(path, config['data']['dataset'], config['data']['feature'], 'test', ToUndirected())
        
        all_graphs = list(train_dataset) + list(val_dataset) + list(test_dataset)
        
        # Build global batch & incidence matrix H
        H = build_hypergraph(train_dataset, val_dataset, test_dataset, config)
        giant_batch = Batch.from_data_list(all_graphs)
        
        # Prepare transductive split indices and labels
        N_train = len(train_dataset)
        N_val = len(val_dataset)
        N_all = len(all_graphs)
        
        train_idx = torch.arange(0, N_train, dtype=torch.long)
        val_idx = torch.arange(N_train, N_train + N_val, dtype=torch.long)
        test_idx = torch.arange(N_train + N_val, N_all, dtype=torch.long)
        
        y = torch.cat([data.y for data in all_graphs], dim=0)
        
        # Initialize model
        model = UPFDModel(
            config=config,
            in_channels=train_dataset.num_features,
            out_channels=train_dataset.num_classes
        )
        
        # Optimizer
        optimizer = torch.optim.Adam(
            model.parameters(), 
            lr=config['training']['lr'], 
            weight_decay=config['training']['weight_decay']
        )
        
        # Transductive Trainer
        trainer = TransductiveTrainer(
            model=model,
            optimizer=optimizer,
            device=device,
            giant_batch=giant_batch,
            H=H,
            train_idx=train_idx,
            val_idx=val_idx,
            test_idx=test_idx,
            y=y
        )
        
        # Start training
        best_test_acc = trainer.fit(config['training']['epochs'])
        return best_test_acc
    else:
        # Initialize standard GNN/Text model
        model = UPFDModel(
            config=config,
            in_channels=train_dataset.num_features,
            out_channels=train_dataset.num_classes
        )
        
        # Optimizer
        optimizer = torch.optim.Adam(
            model.parameters(), 
            lr=config['training']['lr'], 
            weight_decay=config['training']['weight_decay']
        )
        
        # Trainer
        trainer = Trainer(
            model=model,
            optimizer=optimizer,
            device=device,
            train_loader=train_loader,
            val_loader=val_loader,
            test_loader=test_loader
        )
        
        # Start training
        best_test_acc = trainer.fit(config['training']['epochs'])
        return best_test_acc

def main():
    # Load configuration
    config = load_config('config.yaml')
    run_experiment(config)

if __name__ == "__main__":
    main()



In [ ]:
%%writefile config.yaml
# UPFD Project Configuration

data:
  dataset: "politifact" # choices: ['politifact', 'gossipcop']
  feature: "bert"      # choices: ['profile', 'spacy', 'bert', 'content']
  batch_size: 128
  data_dir: "dataset"

model:
  gnn_type: "SAGE"      # choices: ['GCN', 'GAT', 'SAGE']
  pooling: "max"        # choices: ['max', 'mean', 'add', 'attention']
  hidden_channels: 128
  use_gnn: true
  use_text: true
  # Future extensions
  use_sentiment: false
  # HGFND settings
  use_hgfnd: false      # Set to true to use the HGFND architecture from the paper
  hgfnd_layers: 2
  entity_clusters: 50
  time_decimals: 2


training:
  lr: 0.001
  weight_decay: 0.01
  epochs: 80
  device: "cuda" # 'auto', 'cuda', or 'cpu'




In [ ]:
%%writefile colab_experiments.py
import itertools
import pandas as pd
import time
import torch
import random
import numpy as np
import os

# Ensure this script is run in an environment where main.py can be imported
from main import run_experiment

def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

def main():
    results_file = "fake_news_detection_results.csv"
    
    # Overwrite the CSV at the start of the script run
    df_init = pd.DataFrame(columns=[
        "Seed", "Dataset", "Feature", "Architecture Type", "GNN Type", 
        "Use GNN", "Use Text", "Use HGFND", "Pooling", "Epochs", "Accuracy"
    ])
    df_init.to_csv(results_file, index=False)
    print(f"Created/Reset local results file {results_file} for logging.")
        
    seeds = [42, 2026]
    datasets = ["politifact"]
    features = ["bert"] # Can add "spacy"
    
    # Message Passing (Pooling) Experiments
    # Each configuration: (architecture_name, gnn_type, use_gnn, use_text, use_hgfnd, pooling)
    # Each configuration: (architecture_name, gnn_type, use_gnn, use_text, use_hgfnd)
    experiments_matrix = [
        # GNN-Only with different pooling mechanisms
        ("GNN-Only (SAGE) - Max", "SAGE", True, False, False, "max"),
        ("GNN-Only (SAGE) - Mean", "SAGE", True, False, False, "mean"),
        ("GNN-Only (SAGE) - Add", "SAGE", True, False, False, "add"),
        ("GNN-Only (SAGE) - Attn", "SAGE", True, False, False, "attention"),
        
        # GNN+Text with different pooling mechanisms
        ("GNN+Text (SAGE) - Max", "SAGE", True, True, False, "max"),
        ("GNN+Text (SAGE) - Mean", "SAGE", True, True, False, "mean"),
        ("GNN+Text (SAGE) - Add", "SAGE", True, True, False, "add"),
        ("GNN+Text (SAGE) - Attn", "SAGE", True, True, False, "attention"),
        
        # HGFND with different pooling mechanisms
        ("HGFND (SAGE) - Max", "SAGE", True, True, True, "max"),
        ("HGFND (SAGE) - Mean", "SAGE", True, True, True, "mean"),
        ("HGFND (SAGE) - Add", "SAGE", True, True, True, "add"),
        ("HGFND (SAGE) - Attn", "SAGE", True, True, True, "attention"),
    ]

    for seed in seeds:
        for dataset in datasets:
            for feature in features:
                for arch_name, gnn_type, use_gnn, use_text, use_hgfnd, pooling in experiments_matrix:
                    set_seed(seed)
                    
                    config = {
                        "data": {
                            "dataset": dataset,
                            "feature": feature,
                            "batch_size": 128,
                            "data_dir": "dataset"
                        },
                        "model": {
                            "gnn_type": gnn_type,
                            "hidden_channels": 128,
                            "use_gnn": use_gnn,
                            "use_text": use_text,
                            "use_sentiment": False,
                            "use_cmcg": False,
                            # HGFND parameters
                            "use_hgfnd": use_hgfnd,
                            "pooling": pooling,
                            "hgfnd_layers": 2,
                            "entity_clusters": 50,
                            "time_decimals": 2
                        },
                        "training": {
                            "lr": 0.001,
                            "weight_decay": 0.01,
                            "epochs": 80, # Optimized epoch length for comparative evaluation
                            "device": "cuda"
                        }
                    }
                    
                    print(f"\n--- Running Experiment: {arch_name} ---")
                    print(f"Seed: {seed}, Dataset: {dataset}, Feature: {feature}")
                    
                    try:
                        acc = run_experiment(config)
                        acc_val = float(acc)
                        print(f"Achieved Accuracy: {acc_val:.4f}")
                        
                        # Log to CSV
                        row_df = pd.DataFrame([{
                            "Seed": seed, "Dataset": dataset, "Feature": feature, 
                            "Architecture Type": arch_name, "GNN Type": gnn_type, 
                            "Use GNN": use_gnn, "Use Text": use_text, "Use HGFND": use_hgfnd,
                            "Pooling": pooling,
                            "Epochs": config['training']['epochs'], "Accuracy": acc_val
                        }])
                        row_df.to_csv(results_file, mode='a', header=False, index=False)
                            
                    except Exception as e:
                        print(f"Experiment failed: {e}")
                        row_df = pd.DataFrame([{
                            "Seed": seed, "Dataset": dataset, "Feature": feature, 
                            "Architecture Type": arch_name, "GNN Type": gnn_type, 
                            "Use GNN": use_gnn, "Use Text": use_text, "Use HGFND": use_hgfnd,
                            "Pooling": pooling,
                            "Epochs": config['training']['epochs'], "Accuracy": f"ERROR: {str(e)}"
                        }])
                        row_df.to_csv(results_file, mode='a', header=False, index=False)

if __name__ == "__main__":
    main()


# 1. Install Dependencies

In [ ]:
!pip install torch_geometric PyYAML gspread

# 2. Download and Extract Dataset
Fixes the 404 error from PyTorch Geometric.

In [ ]:
!mkdir -p dataset/politifact/raw
!curl -L -o dataset/politifact/raw/data.zip "https://data.pyg.org/datasets/upfd_politifact.zip"
!cd dataset/politifact/raw && unzip -o data.zip && rm data.zip

# 3. Run Experiments
This will authenticate with Google Sheets and start the training loops.

In [ ]:
!python colab_experiments.py